# Deploy BGE-M3 Embedding Model on SageMaker with Scale-to-Zero

This notebook deploys the [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3) multilingual embedding model on a **GPU-powered SageMaker real-time endpoint** that can **scale to zero** when idle — meaning you only pay when the endpoint is actually serving requests.

### Key Concepts

| Concept | What it means |
|---|---|
| **Model** | A SageMaker resource that points to a container image + model artifacts. Think of it as "what to run". |
| **Endpoint** | The actual compute infrastructure (EC2 instances) that hosts your model. Think of it as "where to run". |
| **Endpoint Config** | A blueprint that tells the endpoint what instance type to use, how many instances, and scaling rules. |
| **Inference Component (IC)** | A layer on top of the endpoint that maps a model to compute resources. This is what enables scale-to-zero — SageMaker scales IC copies up/down and manages instances automatically. |
| **Scale to Zero** | When there are no requests for a while, SageMaker removes all instances (cost = $0). When a new request arrives, it spins up an instance automatically (cold start takes a few minutes). |

## Step 0: Install Dependencies

In [ ]:
!pip install -U sagemaker huggingface_hub -q

## Step 1: Setup

We initialize:
- **SageMaker session & role** — the IAM role gives SageMaker permission to pull containers, read S3, etc.
- **Boto3 clients** — low-level AWS SDK clients for SageMaker, Auto Scaling, and CloudWatch (needed for scale-to-zero setup).
- **Configuration constants** — endpoint name, instance type, and scaling timeouts you can customize.

In [ ]:
import sagemaker
import boto3
import json
import time

role = sagemaker.get_execution_role()
sess = sagemaker.Session()
sm_client = boto3.client("sagemaker")
aas_client = boto3.client("application-autoscaling")
cw_client = boto3.client("cloudwatch")

ENDPOINT_NAME = "bge-m3-embedding"
IC_NAME = "bge-m3-ic"
INSTANCE_TYPE = "ml.g5.xlarge"       # GPU instance (24GB VRAM, good for embedding models)
SCALE_IN_COOLDOWN = 600              # seconds idle before scaling to zero (10 min)
SCALE_OUT_COOLDOWN = 300             # seconds to wait before allowing another scale-out

## Step 2: Create the Model

This tells SageMaker **what** to run:
- `HF_MODEL_ID` — the Hugging Face Hub model ID. SageMaker will download it automatically at startup.
- `HF_TASK` — tells the HuggingFace inference container what pipeline to use. `feature-extraction` returns embeddings.
- `transformers_version` / `pytorch_version` — determines which pre-built Deep Learning Container (DLC) image to use.

We use `prepare_container_def()` + `create_model()` (instead of the simpler `.deploy()`) because we need the low-level API to set up inference components later.

In [ ]:
from sagemaker.huggingface import HuggingFaceModel

model = HuggingFaceModel(
    env={
        "HF_MODEL_ID": "BAAI/bge-m3",
        "HF_TASK": "feature-extraction",
    },
    role=role,
    transformers_version="4.37.0",
    pytorch_version="2.1.0",
    py_version="py310",
    sagemaker_session=sess,
)

container = model.prepare_container_def(instance_type=INSTANCE_TYPE)

sm_client.create_model(
    ModelName=ENDPOINT_NAME,
    PrimaryContainer=container,
    ExecutionRoleArn=role,
)
print(f"Model '{ENDPOINT_NAME}' created.")

## Step 3: Create the Endpoint

This tells SageMaker **where** to run the model. Two parts:

**Endpoint Config** — the blueprint:
- `InstanceType` — the hardware to use (ml.g5.xlarge = 1 GPU, 24GB VRAM)
- `InitialInstanceCount: 1` — start with 1 instance
- `ManagedInstanceScaling` — **this is the key setting for scale-to-zero**:
  - `MinInstanceCount: 0` — allows SageMaker to remove ALL instances when idle
  - `MaxInstanceCount: 1` — cap at 1 instance (increase for higher throughput)
- `RoutingConfig` — how to distribute requests across instances

**Endpoint** — the actual deployment. Creating it provisions the instance and starts the container. This takes a few minutes.

In [ ]:
sm_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_NAME,
    ProductionVariants=[
        {
            "VariantName": "default",
            "InstanceType": INSTANCE_TYPE,
            "InitialInstanceCount": 1,
            "ManagedInstanceScaling": {
                "Status": "ENABLED",
                "MinInstanceCount": 0,
                "MaxInstanceCount": 1,
            },
            "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
        }
    ],
)

sm_client.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_NAME,
)

print(f"Creating endpoint '{ENDPOINT_NAME}'... (this takes ~5-10 min)")
t0 = time.time()
waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=ENDPOINT_NAME, WaiterConfig={"Delay": 30, "MaxAttempts": 40})
endpoint_startup = time.time() - t0
print(f"Endpoint in service! Startup time: {endpoint_startup:.1f}s ({endpoint_startup/60:.1f} min)")

## Step 4: Create the Inference Component

An **Inference Component (IC)** maps a model to compute resources on the endpoint. This is required for scale-to-zero.

Why not just use the endpoint directly?
- Without ICs, you can only scale instances (min 1 = always paying).
- With ICs, SageMaker scales **model copies**. When copies = 0, it releases the instance.
- ICs also let you host **multiple models on one endpoint** (not used here, but a nice bonus).

Key settings:
- `NumberOfAcceleratorDevicesRequired: 1` — this model needs 1 GPU
- `MinMemoryRequiredInMb: 1024` — minimum CPU memory
- `CopyCount: 1` — start with 1 copy of the model

In [ ]:
sm_client.create_inference_component(
    InferenceComponentName=IC_NAME,
    EndpointName=ENDPOINT_NAME,
    VariantName="default",
    Specification={
        "ModelName": ENDPOINT_NAME,
        "ComputeResourceRequirements": {
            "NumberOfAcceleratorDevicesRequired": 1,
            "MinMemoryRequiredInMb": 1024,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)

print(f"Creating inference component '{IC_NAME}'...")
t0 = time.time()
while True:
    resp = sm_client.describe_inference_component(InferenceComponentName=IC_NAME)
    status = resp["InferenceComponentStatus"]
    print(f"Status: {status}")
    if status == "InService":
        break
    elif status == "Failed":
        raise RuntimeError(f"Inference component failed: {resp}")
    time.sleep(30)
ic_startup = time.time() - t0
print(f"Inference component ready! Startup time: {ic_startup:.1f}s ({ic_startup/60:.1f} min)")
print(f"Total startup (endpoint + IC): {endpoint_startup + ic_startup:.1f}s ({(endpoint_startup + ic_startup)/60:.1f} min)")

## Step 5: Configure Auto Scaling (Scale to Zero)

This is where the magic happens. We set up **4 things**:

### 1. Register Scalable Target
Tell AWS Auto Scaling that our inference component can be scaled, with `MinCapacity=0` (allows zero copies).

### 2. Target Tracking Policy (scale IN → zero)
Monitors `InvocationsPerCopy`. When this drops to 0 (no traffic) and stays there for `SCALE_IN_COOLDOWN` seconds, it scales the IC copies down to 0. SageMaker then releases the instance.

### 3. Step Scaling Policy (scale OUT → from zero)
Defines the action: "add 1 copy when triggered". But this policy needs something to trigger it...

### 4. CloudWatch Alarm (the trigger)
Watches the `NoCapacityInvocationFailures` metric. When a request hits the endpoint but there are 0 instances, SageMaker emits this metric. The alarm fires → triggers the step policy → provisions an instance → loads the model.

In [ ]:
resource_id = f"inference-component/{IC_NAME}"

# 1. Register scalable target with min=0
aas_client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:inference-component:DesiredCopyCount",
    MinCapacity=0,
    MaxCapacity=1,
)

# 2. Target tracking policy — scales IN to zero after idle
aas_client.put_scaling_policy(
    PolicyName=f"{IC_NAME}-target-tracking",
    PolicyType="TargetTrackingScaling",
    ResourceId=resource_id,
    ServiceNamespace="sagemaker",
    ScalableDimension="sagemaker:inference-component:DesiredCopyCount",
    TargetTrackingScalingPolicyConfiguration={
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerInferenceComponentInvocationsPerCopy"
        },
        "TargetValue": 1,
        "ScaleInCooldown": SCALE_IN_COOLDOWN,
        "ScaleOutCooldown": SCALE_OUT_COOLDOWN,
    },
)

# 3. Step scaling policy — scales OUT from zero
step_resp = aas_client.put_scaling_policy(
    PolicyName=f"{IC_NAME}-scale-from-zero",
    PolicyType="StepScaling",
    ResourceId=resource_id,
    ServiceNamespace="sagemaker",
    ScalableDimension="sagemaker:inference-component:DesiredCopyCount",
    StepScalingPolicyConfiguration={
        "AdjustmentType": "ChangeInCapacity",
        "MetricAggregationType": "Maximum",
        "Cooldown": 60,
        "StepAdjustments": [{"MetricIntervalLowerBound": 0, "ScalingAdjustment": 1}],
    },
)

# 4. CloudWatch alarm to trigger scale-out from zero
cw_client.put_metric_alarm(
    AlarmName=f"{IC_NAME}-no-capacity-alarm",
    AlarmActions=[step_resp["PolicyARN"]],
    MetricName="NoCapacityInvocationFailures",
    Namespace="AWS/SageMaker",
    Dimensions=[{"Name": "InferenceComponentName", "Value": IC_NAME}],
    Statistic="Sum",
    Period=60,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=1,
    ComparisonOperator="GreaterThanThreshold",
)

print(f"Auto scaling configured!")
print(f"  Scale to zero after: {SCALE_IN_COOLDOWN}s ({SCALE_IN_COOLDOWN//60} min) of no invocations")
print(f"  Scale out cooldown:  {SCALE_OUT_COOLDOWN}s ({SCALE_OUT_COOLDOWN//60} min)")

## Step 6: Test Inference

Send sample texts to the endpoint and get back embedding vectors.

Note: When using inference components, you must pass `InferenceComponentName` in the `invoke_endpoint` call so SageMaker knows which model to route the request to.

The response is a list of token-level embeddings per input. We take `[0]` (the CLS token) as the sentence embedding — this is standard for BERT-style models like BGE-M3.

In [ ]:
runtime = boto3.client("sagemaker-runtime")

payload = {"inputs": ["What is BGE-M3?", "BGE-M3 is a multilingual embedding model."]}

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    InferenceComponentName=IC_NAME,
    ContentType="application/json",
    Body=json.dumps(payload),
)

result = json.loads(response["Body"].read())
embeddings = [r[0] for r in result]
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimension: {len(embeddings[0])}")

## Step 7: Measure Cold Start from Zero

⚠️ **Run this cell AFTER the endpoint has scaled to zero** (wait for `SCALE_IN_COOLDOWN` with no requests).

This measures how long it takes to go from **zero instances → serving a request**. The process:
1. Send a request → it fails because there are no instances
2. The failure triggers the CloudWatch alarm → step scaling policy → SageMaker provisions an instance
3. We keep retrying every 15s until we get a successful response
4. The total time is your **cold start latency**

Expect **~4-8 minutes** for ml.g5.xlarge with BGE-M3 (instance provisioning + container start + model download & loading).

In [ ]:
from botocore.exceptions import ClientError

payload = json.dumps({"inputs": ["cold start test"]})

print("Sending request to trigger scale-out from zero...")
t0 = time.time()

while True:
    try:
        response = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            InferenceComponentName=IC_NAME,
            ContentType="application/json",
            Body=payload,
        )
        cold_start = time.time() - t0
        print(f"\nCold start time: {cold_start:.1f}s ({cold_start/60:.1f} min)")
        break
    except ClientError as e:
        elapsed = time.time() - t0
        print(f"  [{elapsed:.0f}s] Not ready yet: {e.response['Error']['Code']}")
        time.sleep(15)

## Step 8: Cleanup

⚠️ **Important**: Delete resources in the correct order to avoid errors:
1. CloudWatch alarm & auto scaling (otherwise scaling may fight your deletion)
2. Inference component (must be deleted before the endpoint)
3. Endpoint → Endpoint config → Model

Uncomment and run when you're done to stop all charges.

In [ ]:
# # 1. Remove auto scaling
# cw_client.delete_alarms(AlarmNames=[f"{IC_NAME}-no-capacity-alarm"])
# aas_client.deregister_scalable_target(
#     ServiceNamespace="sagemaker",
#     ResourceId=f"inference-component/{IC_NAME}",
#     ScalableDimension="sagemaker:inference-component:DesiredCopyCount",
# )
#
# # 2. Delete inference component (wait for it to finish)
# sm_client.delete_inference_component(InferenceComponentName=IC_NAME)
# time.sleep(60)
#
# # 3. Delete endpoint, config, and model
# sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
# sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
# sm_client.delete_model(ModelName=ENDPOINT_NAME)
# print("All resources deleted.")